# 🏗️ Notebook 1: Collaborative Whiteboard — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎨 What we're designing

A real-time collaborative whiteboard — think **Miro**, **Figma (FigJam)**, **Excalidraw**, **tldraw**, or **Google Jamboard**.

Many people draw, move, and delete shapes on the same canvas *at the same time*. Every edit should appear on everyone else's screen within a blink (~100 ms), and in the end everyone must see **exactly the same canvas**, even if their network dropped for a minute.

> **Beginner mental model:** imagine a shared Google Doc, but instead of typing letters, users drag rectangles and scribble lines. The same ideas apply (edits merge, late joiners see current state, offline edits reconcile), but the *unit of edit* is a shape, not a character.


## ✅ Requirements

### Functional
- Draw / move / delete shapes in real time.
- See other users' **cursors and selections** (presence).
- **Late joiners** load the current state of the board.
- **Offline edits** merge back correctly when the user reconnects.
- **Undo / redo** per user.

### Non-functional
- **Convergence** — all clients end up with the *same* canvas (eventual consistency).
- **Fan-out latency** — an edit reaches other users in < 100 ms p50.
- **Availability** — tolerate brief network partitions; never show a blank or wrong canvas.
- **Scalability** — 100k+ concurrent boards, 1M+ open WebSocket connections.
- **Durability** — boards survive server restarts; don't lose strokes.


## 🔢 Back-of-envelope

Numbers are intentionally round so they are easy to reason about — real production numbers vary.

| Quantity | Estimate | Why |
|---|---|---|
| Concurrent boards | 100,000 | steady-state during peak hours |
| Avg users / board | 5 | meetings, classrooms, teams |
| Concurrent WS connections | 500,000 | boards × users |
| Edits / sec / board | 10 | drags produce many small ops |
| Total edits / sec | 1,000,000 | 100k × 10 |
| Bytes / edit | ~300 B | id + coords + color + timestamps |
| Outbound bandwidth | 300 MB/s | 1M × 300 B |

👉 **Implication:** one WebSocket server can hold ~50k connections, so we need ~10 nodes *just for fan-out*, and they must share state (pub/sub) because any user of a board may be on any node.


## 🧱 High-level architecture

```
 ┌──────────┐   WebSocket    ┌─────────────┐    ┌───────────────┐
 │ Browser  │ ─────────────► │ WS Gateway  │──► │ Room Service  │
 └──────────┘                │ (stateless) │    │  (per board)  │
                             └──────┬──────┘    └──────┬────────┘
                                    │                  │
                                    ▼                  ▼
                         ┌─────────────────┐   ┌────────────────┐
                         │ Redis Pub/Sub   │   │ CRDT / Op Log  │
                         │ (board:<id>)    │   │   (per board)  │
                         └────────┬────────┘   └──────┬─────────┘
                                  │                   │
                                  ▼                   ▼
                         other WS Gateways     Snapshot store (S3)
                         (cross-node fan-out)   taken every ~1k ops
```

### Responsibilities
- **WS Gateway** — terminates WebSockets. Stateless so we can autoscale. Just a relay.
- **Room Service** — validates ops for a board, assigns a Lamport clock, writes to the op log, publishes to pub/sub.
- **Redis Pub/Sub** — broadcasts ops to every gateway that has at least one subscriber on that board.
- **CRDT / Op Log** — durable per-board store. Replaying ops reconstructs the board.
- **Snapshot store** — periodic full state so new joiners don't replay millions of ops.


## 🏚️ → 🏛️ Bad → Better → Best architecture

A good interview answer **explains what you rejected and why**, not just the final picture.

### 🏚️ Naive v0 — single server, broadcast to all
- One process keeps an in-memory map `board_id → set[WebSocket]`.
- When a user sends an op, server pushes it to every other socket in that room.

**Why it fails:**
- Single point of failure — restart = everyone disconnects and loses in-flight ops.
- One box can't hold millions of sockets.
- No durability — reload the page and your whiteboard is gone.
- No offline merge — the server *is* the source of truth; if two users edit while partitioned, one wins arbitrarily (lost update).

### 🏗️ v1 — sharded WS gateways + pub/sub
- Add many WS gateways behind a load balancer.
- Use Redis Pub/Sub so any node can fan out to sockets on any other node.
- Persist ops to a database (append-only op log).

**Better, but still:**
- Pub/Sub is fire-and-forget — if a gateway misses a message during restart, clients desync.
- Late joiners still have to replay the whole op log.
- Concurrent edits still race unless we add logical clocks.

### 🏛️ v2 — CRDT + snapshots + resumable streams
- Model the board as a **CRDT** (we use a Last-Writer-Wins map in notebook 3). Concurrent edits merge deterministically — no central serialization needed.
- **Snapshots** every N ops → new joiners download snapshot + replay tail.
- Clients keep a **since-cursor** (last seen Lamport). On reconnect they ask "give me ops > cursor", and pub/sub is just an optimization on top of a durable op log.
- Presence (cursors) is **ephemeral** — pub/sub only, never persisted.

This is roughly what **Figma**, **Liveblocks**, **Yjs**, and **Automerge** converge on.


## 🌍 Real-world references

| Product | Sync approach | Notes |
|---|---|---|
| **Figma** | Custom CRDT-like tree, server-authoritative tie-breaking | [Blog: How Figma's multiplayer works](https://www.figma.com/blog/how-figmas-multiplayer-technology-works/) |
| **Miro** | Op-based with server reconciliation | Large boards → viewport streaming |
| **Excalidraw** | Server-broadcast + optimistic local state | [excalidraw.com](https://excalidraw.com) — OSS, great to read |
| **tldraw** | Uses `@tldraw/sync` on top of a CRDT store | OSS |
| **Google Docs / Sheets** | Operational Transform (OT) | Older but battle-tested approach |
| **Yjs / Automerge** | General-purpose CRDT libraries | Drop-in for many apps |

We'll implement a **tiny CRDT** in notebook 3 to see *why* this works.


## 🧭 What's next

- **Notebook 2** — the data model (`Shape`, `Op`, `Presence`) and HTTP / WebSocket APIs.
- **Notebook 3** — deep dive: Lamport clocks, LWW CRDT merge, naive-vs-CRDT concurrency demo, pub/sub fan-out, snapshots, and presence TTL.
